# GASP — Catalog Exploration
**Gaia Asteroid Spectral Pipeline v1**

This notebook explores `gasp_catalog_v1.parquet`:
19,190 asteroids with Gaia DR3 NUV-corrected reflectance
spectra, SDSS photometry, family membership, and derived
spectral features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# Load catalog
df = pd.read_parquet('../data/final/gasp_catalog_v1.parquet')

print(f"Asteroids: {len(df):,}")
print(f"Columns:   {df.shape[1]}")
print(f"\nCoverage:")
print(f"  Family:   {df['family'].notna().sum():,} "
      f"({df['family'].notna().mean()*100:.1f}%)")
print(f"  Taxonomy: {df['taxonomy'].notna().sum():,} "
      f"({df['taxonomy'].notna().mean()*100:.1f}%)")
print(f"  Albedo:   {df['albedo'].notna().sum():,} "
      f"({df['albedo'].notna().mean()*100:.1f}%)")

BANDS = [374,418,462,506,550,594,638,682,
         726,770,814,858,902,946,990,1034]
REFL_COLS = [f'refl_{b}' for b in BANDS]
BAND_UM = [b/1000 for b in BANDS]

## 1. Sample spectra by SDSS complex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, complex_type, color, title in [
    (axes[0], 'C-complex', 'steelblue', 'C-complex asteroids'),
    (axes[1], 'S-complex', 'tomato',    'S-complex asteroids'),
]:
    sample = df[df['sdss_complex'] == complex_type].sample(
        min(30, len(df[df['sdss_complex'] == complex_type])),
        random_state=42
    )
    for _, row in sample.iterrows():
        ax.plot(BAND_UM, row[REFL_COLS].values,
                color=color, alpha=0.2, lw=1)
    
    # Mean spectrum
    mean_spec = df[df['sdss_complex'] == complex_type][REFL_COLS].mean()
    ax.plot(BAND_UM, mean_spec.values,
            color='black', lw=2.5, label='mean')
    
    ax.axvline(0.55, color='gray', ls='--', alpha=0.4)
    ax.axvspan(0.374, 0.506, alpha=0.07, color='purple',
               label='NUV corrected')
    ax.set_xlabel('Wavelength (µm)', fontsize=12)
    ax.set_ylabel('Reflectance (norm. 550 nm)', fontsize=12)
    ax.set_title(f'{title} (n={len(sample)})', fontsize=13)
    ax.set_ylim(0.5, 1.8)
    ax.legend(fontsize=10)

plt.suptitle('Gaia DR3 NUV-corrected spectra — GASP v1',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/01_spectra_C_vs_S.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/01_spectra_C_vs_S.png")

## 2. NUV slope s1 distribution by complex type

s1 = slope 374–418 nm (µm⁻¹)
Negative s1 = UV absorption (hydration indicator in primitives)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'C-complex': 'steelblue',
          'S-complex': 'tomato',
          'ambiguous': 'gray'}

for complex_type, color in colors.items():
    subset = df[
        (df['sdss_complex'] == complex_type) &
        (df['s1'].notna())
    ]['s1']
    if len(subset) > 10:
        ax.hist(subset, bins=60, alpha=0.5,
                color=color, label=f'{complex_type} (n={len(subset):,})',
                range=(-20, 15), density=True)

ax.axvline(0, color='black', ls='--', lw=1.5,
           label='s1 = 0')
ax.set_xlabel('s1 — NUV slope 374–418 nm (µm⁻¹)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('NUV slope distribution by SDSS complex — GASP v1',
             fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../figures/02_s1_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/02_s1_distribution.png")

## 3. a* color vs NUV slope (s1)

The key GASP diagnostic: SDSS a* separates S from C,
NUV slope s1 measures UV absorption.
Primitive (C-complex) asteroids should show more negative s1.

In [ ]:
subset = df[df['s1'].notna() & df['a_star'].notna()].copy()

# A&A two-column width = 18 cm; fonts >= 12 pt; DPI 300
fig, ax = plt.subplots(figsize=(18 / 2.54, 5))

has_albedo = subset['albedo'].notna()

sc = ax.scatter(
    subset.loc[has_albedo, 'a_star'],
    subset.loc[has_albedo, 's1'].clip(-25, 15),
    c=subset.loc[has_albedo, 'albedo'],
    cmap='RdYlGn', alpha=0.4, s=15,
    vmin=0, vmax=0.4
)
ax.scatter(
    subset.loc[~has_albedo, 'a_star'],
    subset.loc[~has_albedo, 's1'].clip(-25, 15),
    color='lightgray', alpha=0.2, s=8
)

cb = plt.colorbar(sc, ax=ax)
cb.set_label('Albedo $p_V$ (NEOWISE)', fontsize=12)
cb.ax.tick_params(labelsize=12)
ax.axvline(-0.1, color='steelblue', ls='--', lw=1.5,
           alpha=0.7, label='C/ambiguous boundary ($a^*=-0.10$)')
ax.axvline(0.1, color='tomato', ls='--', lw=1.5,
           alpha=0.7, label='ambiguous/S boundary ($a^*=+0.10$)')
ax.axhline(0, color='black', ls=':', lw=1.5, alpha=0.5)

ax.set_xlabel(r'SDSS $a^*$ color index', fontsize=12)
ax.set_ylabel(r'$s_1$ — NUV slope 374–418 nm ($\mu$m$^{-1}$)', fontsize=12)
ax.tick_params(labelsize=12)
ax.legend(fontsize=12)
ax.set_xlim(-0.5, 0.5)
plt.tight_layout()
plt.savefig('../figures/03_astar_vs_s1.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: figures/03_astar_vs_s1.png")

## 4. Top families — spectral diversity

In [ ]:
top_families = (df['family'].value_counts()
                .head(8).index.tolist())

fig, axes = plt.subplots(2, 4, figsize=(16, 8),
                          sharey=True, sharex=True)

for ax, family in zip(axes.flatten(), top_families):
    members = df[df['family'] == family]
    
    # Individual spectra (max 50)
    sample = members.sample(min(50, len(members)),
                             random_state=42)
    for _, row in sample.iterrows():
        ax.plot(BAND_UM, row[REFL_COLS].values,
                color='steelblue', alpha=0.15, lw=0.8)
    
    # Mean
    mean_spec = members[REFL_COLS].mean()
    ax.plot(BAND_UM, mean_spec.values,
            color='black', lw=2, label='mean')
    
    ax.axvline(0.55, color='gray', ls='--', alpha=0.3)
    ax.set_title(f'{family}\n(n={len(members):,})',
                 fontsize=10)
    ax.set_ylim(0.5, 2.0)

for ax in axes[1]:
    ax.set_xlabel('Wavelength (µm)', fontsize=9)
for ax in axes[:, 0]:
    ax.set_ylabel('Reflectance', fontsize=9)

plt.suptitle('Mean spectra of top 8 asteroid families — GASP v1',
             fontsize=13)
plt.tight_layout()
plt.savefig('../figures/04_family_spectra.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/04_family_spectra.png")

## Summary

| Metric | Value |
|--------|-------|
| Asteroids | 19,190 |
| Spectral bands | 16 (NUV-corrected) |
| Family coverage | 39.3% |
| S-complex | ~43% |
| C-complex | ~12% |
| Reliable NUV (s1) | ~57% (s1 not NaN) |

**Key observations from this catalog:**
- S-complex asteroids show systematically redder spectra
- C-complex asteroids show more NUV absorption (negative s1)
- Family members show tighter spectral clustering than background

In [ ]:
print("=== GASP Exploration Complete ===")
print(f"Figures saved to: figures/")
print("  01_spectra_C_vs_S.png")
print("  02_s1_distribution.png")
print("  03_astar_vs_s1.png")
print("  04_family_spectra.png")